# 11 — Hardware Result Extraction and QEM Evaluation

This notebook converts completed IBM Quantum SamplerV2 jobs into publication-ready raw-count datasets and evaluates raw versus mitigated distributions.

IBM's current examples retrieve counts from `result[0].data.<register>.get_counts()`. The register name depends on the circuit, so this notebook discovers a measurement register instead of assuming one fixed name. citeturn0search5turn0search9

**No hardware jobs are submitted here.**


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if (ROOT / "Adaptive_QEM_IBM").exists() and not (ROOT / "data").exists():
    ROOT = ROOT / "Adaptive_QEM_IBM"
sys.path.insert(0, str(ROOT))

from analysis.metrics import normalize_counts, total_variation_distance, distribution_fidelity, success_probability
from analysis.statistics import wilson_interval
from mitigation.readout_mitigation import mitigate_counts
from mitigation.zne import linear_extrapolate_zero_noise

print("ROOT:", ROOT)


## 1. Load completed job records


In [ ]:
JOB_DIR = ROOT / "data/hardware/jobs"
if not JOB_DIR.exists():
    raise FileNotFoundError("No job directory found. Complete Notebook 10 first.")

job_files = sorted(JOB_DIR.glob("*.json"))
print("Job metadata files:", len(job_files))
for f in job_files:
    print(f.name)


## 2. Job-result extraction helper

The common SamplerV2 pattern is `pub_result.data.<register>.get_counts()`. If a circuit contains multiple classical registers, inspect the returned register names and select the benchmark's measurement register. citeturn0search9


In [ ]:
def extract_counts_from_pub(pub_result):
    data = getattr(pub_result, "data", None)
    if data is None:
        raise RuntimeError("Sampler result has no data field.")

    if hasattr(data, "meas") and hasattr(data.meas, "get_counts"):
        return dict(data.meas.get_counts())

    for name in dir(data):
        if name.startswith("_"):
            continue
        try:
            obj = getattr(data, name)
        except Exception:
            continue
        if hasattr(obj, "get_counts"):
            return dict(obj.get_counts())

    raise RuntimeError("No classical BitArray with get_counts() was found.")


## 3. Retrieve job results


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
results = []

for jf in job_files:
    meta = json.loads(jf.read_text(encoding="utf-8"))
    job_id = meta.get("job_id")
    if not job_id:
        continue

    print("Retrieving:", job_id)
    job = service.retrieve_job(job_id)
    result = job.result()

    names = meta.get("circuits", [])
    for i, pub in enumerate(result):
        counts = extract_counts_from_pub(pub)
        name = names[i] if i < len(names) else f"pub_{i}"
        results.append({
            "job_id": job_id,
            "backend": meta.get("backend"),
            "strategy": meta.get("strategy"),
            "circuit": name,
            "shots": meta.get("shots"),
            "counts": counts,
        })

print("Retrieved circuit results:", len(results))


## 4. Save normalized raw hardware results


In [ ]:
raw_rows = []
for r in results:
    probs = normalize_counts(r["counts"])
    raw_rows.append({
        "job_id": r["job_id"],
        "backend": r["backend"],
        "strategy": r["strategy"],
        "circuit": r["circuit"],
        "shots": r["shots"],
        "counts_json": json.dumps(r["counts"], sort_keys=True),
        "probabilities_json": json.dumps(probs, sort_keys=True),
    })

raw_df = pd.DataFrame(raw_rows)
out = ROOT / "data/hardware/extracted"
out.mkdir(parents=True, exist_ok=True)
raw_df.to_csv(out / "ibm_sampler_v2_extracted_results.csv", index=False)
display(raw_df.head())


## 5. Readout-error mitigation

The correction uses the assignment probabilities captured in Notebook 09. Raw hardware counts remain unchanged; the corrected distribution is stored separately.


In [ ]:
cal_file = ROOT / "data/calibration/ibm_kingston_calibration_snapshot.csv"
if not cal_file.exists():
    raise FileNotFoundError("Calibration snapshot missing.")

cal = pd.read_csv(cal_file)

def calibration_vectors_for_n(n):
    c = cal.head(n)
    if len(c) < n:
        raise ValueError("Calibration snapshot does not contain enough qubits.")
    e01 = c["prob_meas1_prep0"].tolist()
    e10 = c["prob_meas0_prep1"].tolist()
    if any(pd.isna(e01)) or any(pd.isna(e10)):
        raise ValueError("Required readout assignment probabilities are missing.")
    return e01, e10

mitigated_rows = []
for r in results:
    if r["strategy"] not in ("readout_mitigation", "combined"):
        continue

    n = len(next(iter(r["counts"])).replace(" ", ""))
    e01, e10 = calibration_vectors_for_n(n)
    corrected = mitigate_counts(r["counts"], e01, e10)

    mitigated_rows.append({
        "job_id": r["job_id"],
        "circuit": r["circuit"],
        "shots": r["shots"],
        "mitigation": "readout_mitigation",
        "probabilities_json": json.dumps(corrected, sort_keys=True),
    })

ro_df = pd.DataFrame(mitigated_rows)
ro_df.to_csv(out / "readout_mitigated_results.csv", index=False)
display(ro_df.head())


## 6. Deterministic benchmark success evaluation

For deterministic benchmarks, success states should come from the verified ideal circuit definition. For non-deterministic benchmarks such as QRNG, use distribution fidelity/TVD rather than a single-state success score.


In [ ]:
EXPECTED = {
    "Bell_Phi_Plus": ["00", "11"],
    "GHZ_3": ["000", "111"],
    "GHZ_4": ["0000", "1111"],
    "GHZ_5": ["00000", "11111"],
}

rows = []
for r in results:
    if r["circuit"] in EXPECTED:
        total = int(sum(r["counts"].values()))
        successes = sum(r["counts"].get(k, 0) for k in EXPECTED[r["circuit"]])
        s = successes / total if total else np.nan
        lo, hi = wilson_interval(successes, total)
        rows.append({
            "job_id": r["job_id"],
            "circuit": r["circuit"],
            "strategy": r["strategy"],
            "success_probability": s,
            "wilson_low": lo,
            "wilson_high": hi,
        })

success_df = pd.DataFrame(rows)
display(success_df)


## 7. ZNE result extraction and extrapolation

After ZNE job retrieval, create `data/hardware/extracted/zne_observables.csv` with:

`circuit, scale_factor, observable`

For deterministic circuits, `observable` can be success probability. The notebook extrapolates the measured observable to scale factor zero.


In [ ]:
zne_file = out / "zne_observables.csv"

if zne_file.exists():
    zne = pd.read_csv(zne_file)
    zne_rows = []

    for circuit, g in zne.groupby("circuit"):
        g = g.sort_values("scale_factor")
        z0 = linear_extrapolate_zero_noise(
            g["scale_factor"].tolist(),
            g["observable"].tolist()
        )
        zne_rows.append({
            "circuit": circuit,
            "zne_extrapolated_observable": z0,
            "scale_factors": ",".join(map(str, g["scale_factor"].tolist())),
        })

    zne_df = pd.DataFrame(zne_rows)
    zne_df.to_csv(out / "zne_extrapolated_results.csv", index=False)
    display(zne_df)
else:
    print("No zne_observables.csv yet. ZNE evaluation remains pending.")


## 8. Final comparison dataset

The final paper dataset should contain raw, readout-mitigated, ZNE, and adaptive-selected results side-by-side, with job IDs and execution overhead retained.


In [ ]:
comparison_file = out / "qem_comparison_results.csv"

if success_df.empty:
    print("No deterministic success results available yet.")
else:
    comparison = success_df.copy()
    comparison["error_probability"] = 1.0 - comparison["success_probability"]
    comparison.to_csv(comparison_file, index=False)
    display(comparison)
    print("Saved:", comparison_file)


## 9. Research-integrity checklist

Before making claims:

- Every hardware number has a real IBM job ID.
- Calibration values correspond to the execution period.
- Raw counts are preserved.
- Mitigation is applied only to appropriate raw results.
- ZNE scale factors are explicitly recorded.
- Shot counts are controlled for comparisons.
- Confidence intervals/repeated runs are reported where available.
- Adaptive selection is evaluated against fixed baselines.
- Simulation and hardware results are clearly separated.
